# M2 — GLiNER (zero-shot, sem fine-tuning)

Segundo braço da comparação do M2 (ver PRD.md, seção 8 e 10). GLiNER é um modelo de NER **zero-shot**: em vez de ter uma lista fixa de labels (como o spaCy), ele recebe os nomes das entidades que você quer encontrar como texto livre, e tenta casar spans do documento com essas descrições — sem precisar treinar nada para começar. Por isso já dá pra rodar contra as 5 entidades diretamente (CPF, RG, CIN, NOME, ENDERECO), diferente do spaCy que não tinha noção nenhuma dos identificadores estruturados.

Isso não substitui o fine-tuning (BERTimbau) como comparação — é um segundo ponto de referência: um modelo **genérico mas já desenhado pra PII** (o checkpoint usado aqui foi fine-tuned pela comunidade especificamente para reconhecimento de PII multilíngue, incluindo a categoria "address" nativamente — mesmo pedigree do modelo usado no paper LegalGuardian).

**Diferença importante em relação ao spaCy:** os rótulos (`GLINER_LABELS` abaixo) são só *texto descritivo* — não são uma API fixa. A qualidade do resultado depende de como você descreve cada entidade, e costuma exigir iteração (é um hiperparâmetro do experimento, documentar no anotacoes.txt se for ajustado).

**Limite de contexto:** GLiNER é baseado em transformer com janela de contexto limitada — páginas longas são processadas em janelas (`TAMANHO_JANELA` caracteres, com sobreposição) para não estourar o limite do modelo nem perder entidades perto do fim da página.

**Pré-requisitos:** `pip install gliner torch` (no venv GERAL do projeto, `.venv`). **Não instalar neste venv o stack de OCR** (`paddlepaddle`/`paddleocr`) — ver a restrição documentada em `requirements_ocr.txt` (conflito de cuDNN entre torch e paddlepaddle-gpu no Windows).

In [ ]:
import json
from pathlib import Path

import torch
from gliner import GLiNER

print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Rodando em CPU — mais lento, mas funcional para este volume de documentos.")

## Configuração

In [ ]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

MODELO_GABARITO = "gemini-2.5-flash"  # mesmo modelo usado no notebook 03, para comparar contra o mesmo gabarito

PASTA_EXTRACAO = RAIZ / "data" / "documento_social" / "03.extracao_ocr_local"
# Gabarito dos documentos reais (via Gemini + alinhamento, notebook 03) e dos sintéticos
# (gerado direto, notebook 00) ficam em subpastas irmãs — juntamos as duas na avaliação.
PASTA_GABARITO_REAIS = RAIZ / "data" / "documento_social" / "04.gabarito_alinhado" / MODELO_GABARITO
PASTA_GABARITO_SINTETICOS = RAIZ / "data" / "documento_social" / "04.gabarito_alinhado" / "sintetico"
PASTA_SAIDA = RAIZ / "data" / "documento_social" / "06.deteccao_gliner"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

GLINER_CHECKPOINT = "urchade/gliner_multi_pii-v1"  # mesma linhagem do modelo usado no LegalGuardian
GLINER_THRESHOLD = 0.5   # confiança mínima para aceitar uma entidade (0-1)
TAMANHO_JANELA = 1200    # caracteres por janela de contexto enviada ao modelo
SOBREPOSICAO_JANELA = 100  # caracteres de sobreposição entre janelas consecutivas

IOU_MINIMO_MATCH = 0.30  # limiar de sobreposição de bbox para considerar uma detecção correta (avaliação)

# rótulo enviado ao GLiNER -> nosso tipo de entidade. O texto do rótulo é um hiperparâmetro:
# ajustar aqui se os resultados vierem ruins, e documentar a mudança no anotacoes.txt.
# CIN fundida com CPF (decisão 2026-09-09, ver PRD.md secao 8) -- nao ha rotulo proprio pra CIN.
GLINER_LABELS = {
    "person": "NOME",
    "address": "ENDERECO",
    "tax identification number": "CPF",
    "identity card number": "RG",
}

print(f"Raiz do projeto: {RAIZ}")
print(f"Extração:               {PASTA_EXTRACAO}")
print(f"Gabarito (reais):       {PASTA_GABARITO_REAIS}")
print(f"Gabarito (sintéticos):  {PASTA_GABARITO_SINTETICOS}")
print(f"Saída:                  {PASTA_SAIDA}")

## Carrega o modelo GLiNER

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GLiNER.from_pretrained(GLINER_CHECKPOINT)
model = model.to(device)
print(f"Modelo carregado em: {device}")

## Reconstrução de texto por página, janelamento e mapeamento de volta para bbox

In [ ]:
def construir_texto_pagina(tokens: list) -> tuple:
    """Concatena os tokens da página em texto corrido (sem normalizar) e retorna
    (texto, mapa) onde mapa[i] = índice do token que ocupa a posição i do texto."""
    partes = []
    mapa = []
    for idx, tok in enumerate(tokens):
        if partes:
            partes.append(" ")
            mapa.append(idx)
        partes.append(tok["text"])
        mapa.extend([idx] * len(tok["text"]))
    return "".join(partes), mapa


def dividir_em_janelas(texto: str, tamanho: int, sobreposicao: int) -> list:
    """Divide o texto em janelas de até `tamanho` caracteres, tentando não cortar
    no meio de uma palavra, com sobreposição entre janelas consecutivas."""
    janelas = []
    n = len(texto)
    inicio = 0
    while inicio < n:
        fim = min(inicio + tamanho, n)
        if fim < n:
            espaco = texto.rfind(" ", inicio, fim)
            if espaco > inicio:
                fim = espaco
        janelas.append((inicio, fim))
        if fim >= n:
            break
        inicio = max(fim - sobreposicao, inicio + 1)
    return janelas


def agrupar_bboxes_por_linha(tokens: list, indices: list) -> list:
    """Une os bboxes dos tokens cobertos, quebrando em uma caixa por linha
    (mesma lógica dos notebooks 03/04 — detecta mudança de linha por salto vertical do bbox)."""
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens[indices[0]]["bbox"]]
    y_ref = tokens[indices[0]]["bbox"][1]
    for i in indices[1:]:
        bbox = tokens[i]["bbox"]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)

    caixas = []
    for g in grupos:
        x0 = min(b[0] for b in g)
        y0 = min(b[1] for b in g)
        x1 = max(b[2] for b in g)
        y1 = max(b[3] for b in g)
        caixas.append([x0, y0, x1, y1])
    return caixas

## Detecção: roda o GLiNER sobre cada documento extraído

In [ ]:
def detectar_pagina(texto: str, mapa: list, tokens: list, pagina: int) -> list:
    labels = list(GLINER_LABELS.keys())
    entidades = []
    vistos = set()  # dedup entre janelas sobrepostas: (start_abs, end_abs, label)

    for inicio, fim in dividir_em_janelas(texto, TAMANHO_JANELA, SOBREPOSICAO_JANELA):
        trecho = texto[inicio:fim]
        previsoes = model.predict_entities(trecho, labels, threshold=GLINER_THRESHOLD)
        for p in previsoes:
            tipo = GLINER_LABELS.get(p["label"])
            if tipo is None:
                continue
            start_abs = inicio + p["start"]
            end_abs = inicio + p["end"]
            chave = (start_abs, end_abs, tipo)
            if chave in vistos:
                continue
            vistos.add(chave)

            indices = sorted(set(mapa[start_abs:end_abs]))
            if not indices:
                continue
            bboxes = agrupar_bboxes_por_linha(tokens, indices)
            entidades.append({
                "type": tipo,
                "text": p["text"],
                "page": pagina,
                "bbox": bboxes,
                "gliner_label": p["label"],
                "score": round(float(p["score"]), 4),
            })
    return entidades


def detectar_documento(extracao: dict) -> dict:
    entidades = []
    for pagina_dados in extracao["pages"]:
        tokens = pagina_dados["tokens"]
        if not tokens:
            continue
        texto, mapa = construir_texto_pagina(tokens)
        entidades.extend(detectar_pagina(texto, mapa, tokens, pagina_dados["page"]))
    return {"document_id": extracao["document_id"], "source_file": extracao["source_file"], "entities": entidades}


arquivos_extracao = sorted(PASTA_EXTRACAO.glob("*.json"))
if not arquivos_extracao:
    raise SystemExit(f"Nenhum JSON de extração encontrado em {PASTA_EXTRACAO}")

for arq in arquivos_extracao:
    extracao = json.loads(arq.read_text(encoding="utf-8"))
    resultado = detectar_documento(extracao)
    destino = PASTA_SAIDA / arq.name
    destino.write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
    contagem = {}
    for e in resultado["entities"]:
        contagem[e["type"]] = contagem.get(e["type"], 0) + 1
    print(f"{arq.name}: {contagem} -> {destino.name}")

print("\nDetecção concluída.")

## Avaliação: Precision/Recall/F1 por tipo de entidade, contra o gabarito alinhado

Mesma lógica de matching por IoU de bbox usada no `notebooks/04.baseline_spacy.ipynb`, para os números serem diretamente comparáveis entre os dois motores.

In [ ]:
def bbox_externo(lista_bboxes: list) -> list:
    if not lista_bboxes:
        return None
    x0 = min(b[0] for b in lista_bboxes)
    y0 = min(b[1] for b in lista_bboxes)
    x1 = max(b[2] for b in lista_bboxes)
    y1 = max(b[3] for b in lista_bboxes)
    return [x0, y0, x1, y1]


def iou(a: list, b: list) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    largura, altura = max(0, xi1 - xi0), max(0, yi1 - yi0)
    inter = largura * altura
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0


def avaliar_documento(previstas: list, gabarito: list, contadores: dict):
    """Atualiza contadores[type] = {'tp':.., 'fp':.., 'fn':..} in-place."""
    previstas_por_pag_tipo = {}
    for e in previstas:
        previstas_por_pag_tipo.setdefault((e["page"], e["type"]), []).append(e)

    usadas = set()
    for g in gabarito:
        tipo = g["type"]
        contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
        candidatos = previstas_por_pag_tipo.get((g["page"], tipo), [])
        bbox_gold = bbox_externo(g.get("bbox") or [])
        melhor_iou, melhor_idx = 0.0, None
        for i, p in enumerate(candidatos):
            if id(p) in usadas:
                continue
            score = iou(bbox_gold, bbox_externo(p.get("bbox") or []))
            if score > melhor_iou:
                melhor_iou, melhor_idx = score, i
        if melhor_idx is not None and melhor_iou >= IOU_MINIMO_MATCH:
            usadas.add(id(candidatos[melhor_idx]))
            contadores[tipo]["tp"] += 1
        else:
            contadores[tipo]["fn"] += 1

    for (pagina, tipo), candidatos in previstas_por_pag_tipo.items():
        contadores.setdefault(tipo, {"tp": 0, "fp": 0, "fn": 0})
        for p in candidatos:
            if id(p) not in usadas:
                contadores[tipo]["fp"] += 1

In [ ]:
TODOS_OS_TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]  # CIN fundida com CPF (decisão 2026-09-09)
contadores = {t: {"tp": 0, "fp": 0, "fn": 0} for t in TODOS_OS_TIPOS}

arquivos_gabarito = sorted(PASTA_GABARITO_REAIS.glob("*.json")) + sorted(PASTA_GABARITO_SINTETICOS.glob("*.json"))
arquivos_gabarito = [f for f in arquivos_gabarito if f.name != "_nao_alinhadas.json"]

for arq_gab in arquivos_gabarito:
    arq_pred = PASTA_SAIDA / arq_gab.name
    if not arq_pred.exists():
        print(f"[AVISO] Sem detecção correspondente para {arq_gab.name}, pulando.")
        continue
    gabarito = json.loads(arq_gab.read_text(encoding="utf-8"))
    previstas = json.loads(arq_pred.read_text(encoding="utf-8"))
    gold_alinhado = [e for e in gabarito["entities"] if e.get("match_type") != "unmatched"]
    avaliar_documento(previstas["entities"], gold_alinhado, contadores)

print(f"{'Tipo':<10}{'TP':<6}{'FP':<6}{'FN':<6}{'Precision':<12}{'Recall':<10}{'F1':<8}")
print("-" * 62)
for tipo in TODOS_OS_TIPOS:
    c = contadores[tipo]
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"{tipo:<10}{tp:<6}{fp:<6}{fn:<6}{precision:<12.3f}{recall:<10.3f}{f1:<8.3f}")

tp_total = sum(c["tp"] for c in contadores.values())
fp_total = sum(c["fp"] for c in contadores.values())
fn_total = sum(c["fn"] for c in contadores.values())
precision_geral = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0.0
recall_geral = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0.0
f1_geral = 2 * precision_geral * recall_geral / (precision_geral + recall_geral) if (precision_geral + recall_geral) else 0.0
print("-" * 62)
print(f"{'GERAL':<10}{tp_total:<6}{fp_total:<6}{fn_total:<6}{precision_geral:<12.3f}{recall_geral:<10.3f}{f1_geral:<8.3f}")